<a href="https://colab.research.google.com/github/KathrynC/LEMURS_simulator/blob/master/diffLudobots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip3 install taichi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 18.4 MB/s eta 0:00:00


In [1]:
from numpy import real
import taichi as ti
import numpy as np

max_steps = 100

dt = 0.01

gravity = -9.8

ground_height = 0.1

startingObjectPositions = []

n_objects = 4

for objectIndex in range(n_objects):

  startingObjectPositions.append( [
    np.random.random() ,
    np.random.random()*0.9 + 0.1] )

real = ti.f32

ti.init( default_fp = real )

vec = lambda: ti.Vector.field(2, dtype=real)

positions = vec()

velocities = vec()

ti.root.dense(ti.i , max_steps).dense(ti.j , n_objects).place(positions)
ti.root.dense(ti.i , max_steps).dense(ti.j , n_objects).place(positions.grad)
ti.root.dense(ti.i , max_steps).dense(ti.j , n_objects).place(velocities)

loss = ti.field( dtype=ti.f32 , shape = () , needs_grad = True)

@ti.kernel
def Compute_Loss():
  loss[None] = positions[max_steps-1,0][1]

def Draw(frameOffset):

  for timeStep in range(0,max_steps):

    gui = ti.GUI("Robot" , (512,512), background_color=0xFFFFFF, show_gui=False)

    gui.line( begin = ( 0 , ground_height) ,
            end = (1,ground_height) ,
            color = 0x0,
            radius = 3)

    for objectIndex in range(n_objects):

        x = positions[timeStep,objectIndex][0]
        y = positions[timeStep,objectIndex][1]

        gui.circle( (x,y) , color = 0x0, radius=7)

    gui.show( 'test' + str(frameOffset + timeStep) + '.png')


def Initialize():

  for objectIndex in range(n_objects):

    positions[0,objectIndex] = startingObjectPositions[objectIndex]

    velocities[0,objectIndex] = [0,-0.1]


def Simulate():

    for timeStep in range(1,max_steps):

      Step_One(timeStep)

@ti.kernel
def Step_One(timeStep: ti.i32):

    for objectIndex in range(n_objects):

          oldPosition = positions[timeStep-1,objectIndex]

          oldVelocity = velocities[timeStep-1,objectIndex] + dt * gravity * ti.Vector([0,1])

          if oldPosition[1] <= ground_height:

            oldVelocity = ti.Vector([0,0])

          newPosition = oldPosition + dt * oldVelocity

          positions[timeStep,objectIndex] = newPosition

          newVelocity = oldVelocity

          velocities[timeStep,objectIndex] = newVelocity

Initialize()

with ti.ad.Tape(loss):

  Simulate()

  Compute_Loss()

os.system("rm *.png")

Draw(0)

startingObjectPositions[0] = startingObjectPositions[0] + 0.1 * positions.grad[0,0]

Initialize()

Simulate()

Draw(max_steps)

ModuleNotFoundError: No module named 'taichi'

In [ ]:
import os

os.system("rm movie.mp4")
os.system(" ffmpeg -i test%d.png movie.mp4")

0

In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open('movie.mp4' , 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML('<video width=512 controls> <source src="%s" type="video/mp4"></video>' % data_url)